# Step 3.2 — Comparative MD Analysis

**Project:** DES Peptide Simulation Study  
**Author:** Ross Gibson  
**Date:** April 2026  
**Phase:** 3 — Comparative MD Analysis

---

## Purpose

Quantify and visualise the differences in peptide motif behaviour across solvents  
and across peptides, using the per-system metrics from Step 3.1.

### Analyses performed

1. **Cross-solvent deltas** for each motif:  
   - ΔSASA = SASA_DES − SASA_water  
   - ΔHydration = coordination_DES − coordination_water  
   - ΔH-bond events (fractional change vs water)  
   - Propagated bootstrap 95% CIs on all deltas  

2. **Effect sizes** (Cohen's d) for SASA and water coordination  

3. **Cross-motif comparison**:  
   - Which motif shows the greatest solvent-dependent accessibility change?  
   - Does reline vs glyceline differentially affect different motif types?  

4. **Publication figures**:  
   - Grouped delta bar charts with CIs  
   - Summary heatmap (all metrics × all conditions)  
   - DES component coordination breakdown  

## Inputs

- `analysis/per_system_metrics.csv` (from Step 3.1)  
- `analysis/rmsd_equilibration_summary.csv` (from Step 3.1)  

## Outputs

- `analysis/comparative_deltas.csv` — all cross-solvent deltas with CIs and effect sizes  
- `analysis/cross_motif_summary.csv` — ranked motif responsiveness  
- `analysis/figures/` — publication-quality comparative figures  

## Runtime estimate

All computations are on pre-computed summary statistics. Expected runtime: < 30 seconds.

---
## 1. Configuration and data loading

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# ── Paths ──────────────────────────────────────────────────────────
PROJECT_DIR = os.path.expanduser('~/des-peptide-study')
ANALYSIS_DIR = os.path.join(PROJECT_DIR, 'analysis')
FIGURES_DIR = os.path.join(ANALYSIS_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Labels ──────────────────────────────────────────────────────────
PEPTIDES = ['GGE', 'CME', 'YIY']
SOLVENTS = ['water', 'reline', 'glyceline']
DES_SOLVENTS = ['reline', 'glyceline']

# Colour scheme (consistent with Step 3.1)
SOLVENT_COLORS = {'water': '#2c7bb6', 'reline': '#d7191c', 'glyceline': '#1a9641'}
PEPTIDE_COLORS = {'GGE': '#e66101', 'CME': '#5e3c99', 'YIY': '#1b9e77'}

print(f'Loading data from: {ANALYSIS_DIR}')

In [ ]:
# ── Load per-system metrics ────────────────────────────────────────
metrics_path = os.path.join(ANALYSIS_DIR, 'per_system_metrics.csv')
df_all = pd.read_csv(metrics_path)

print(f'Loaded: {metrics_path}')
print(f'  Rows: {len(df_all)}, Columns: {len(df_all.columns)}')

# Extract summary rows (motif totals) — these have coordination and H-bond data
df = df_all[df_all['residue'].str.endswith('_total')].copy()
df = df.sort_values(['peptide', 'solvent']).reset_index(drop=True)

print(f'\nSummary rows (motif totals): {len(df)}')
print(df[['system', 'SASA_mean_nm2', 'water_coord_mean', 'Hbond_n_events']].to_string(index=False))

# Also extract per-residue rows for residue-level analysis
df_res = df_all[~df_all['residue'].str.endswith('_total')].copy()
df_res = df_res.sort_values(['peptide', 'solvent', 'residue_idx']).reset_index(drop=True)
print(f'\nPer-residue rows: {len(df_res)}')

---
## 2. Cross-solvent deltas

For each peptide, compute the difference between each DES solvent and water  
for SASA, water coordination, and H-bond events.

**CI propagation:** For the difference of two means with independent bootstrap CIs,  
we approximate the delta CI by summing the half-widths in quadrature:  

$$\text{CI}_{\Delta} = \sqrt{\text{hw}_{\text{DES}}^2 + \text{hw}_{\text{water}}^2}$$

where $\text{hw} = (\text{CI}_{\text{hi}} - \text{CI}_{\text{lo}}) / 2$.  
This assumes approximate normality of the bootstrap distribution, which is  
reasonable at n=2000 replicates.

In [ ]:
def propagate_ci(hw_a, hw_b):
    """Propagate CI half-widths for a difference (a - b)."""
    return np.sqrt(hw_a**2 + hw_b**2)


def cohens_d_from_ci(mean_a, hw_a, mean_b, hw_b):
    """Approximate Cohen's d from means and CI half-widths.
    
    Estimates pooled SD from CI half-widths, assuming a 95% CI
    corresponds to ~1.96 * SE. For bootstrap CIs with large n_frames,
    SE is small relative to SD, so we use:
        SD ~ hw * sqrt(n) / 1.96
    But since we don't know n per group here, we use the CI half-width
    directly as a conservative proxy for the SE, and report the
    signal-to-noise ratio |delta| / CI_delta as an analogous measure.
    
    Instead, we compute a bootstrap-based effect size:
        ES = |mean_a - mean_b| / pooled_hw
    where pooled_hw = sqrt((hw_a^2 + hw_b^2) / 2)
    
    This is interpretable as: how many CI half-widths apart are the means?
    Values > 2 indicate non-overlapping CIs (clear separation).
    """
    delta = abs(mean_a - mean_b)
    pooled_hw = np.sqrt((hw_a**2 + hw_b**2) / 2)
    if pooled_hw == 0:
        return np.inf if delta > 0 else 0.0
    return delta / pooled_hw


# ── Compute deltas ─────────────────────────────────────────────────
delta_rows = []

for pep in PEPTIDES:
    water = df[(df['peptide'] == pep) & (df['solvent'] == 'water')].iloc[0]
    
    for des in DES_SOLVENTS:
        des_row = df[(df['peptide'] == pep) & (df['solvent'] == des)].iloc[0]
        
        # ── SASA delta ────────────────────────────────────────────
        sasa_delta = des_row['SASA_mean_nm2'] - water['SASA_mean_nm2']
        sasa_hw_w = (water['SASA_CI95_hi'] - water['SASA_CI95_lo']) / 2
        sasa_hw_d = (des_row['SASA_CI95_hi'] - des_row['SASA_CI95_lo']) / 2
        sasa_delta_hw = propagate_ci(sasa_hw_w, sasa_hw_d)
        sasa_pct = (sasa_delta / water['SASA_mean_nm2']) * 100
        sasa_es = cohens_d_from_ci(des_row['SASA_mean_nm2'], sasa_hw_d,
                                   water['SASA_mean_nm2'], sasa_hw_w)
        
        # ── Water coordination delta ──────────────────────────────
        wc_delta = des_row['water_coord_mean'] - water['water_coord_mean']
        wc_hw_w = (water['water_coord_CI95_hi'] - water['water_coord_CI95_lo']) / 2
        wc_hw_d = (des_row['water_coord_CI95_hi'] - des_row['water_coord_CI95_lo']) / 2
        wc_delta_hw = propagate_ci(wc_hw_w, wc_hw_d)
        wc_pct = (wc_delta / water['water_coord_mean']) * 100
        wc_es = cohens_d_from_ci(des_row['water_coord_mean'], wc_hw_d,
                                 water['water_coord_mean'], wc_hw_w)
        
        # ── H-bond event count change ─────────────────────────────
        hb_water = water['Hbond_n_events']
        hb_des = des_row['Hbond_n_events']
        hb_delta = hb_des - hb_water
        hb_pct = (hb_delta / hb_water * 100) if hb_water > 0 else np.nan
        
        # ── H-bond median lifetime delta ──────────────────────────
        hb_med_w = water['Hbond_median_ps']
        hb_med_d = des_row['Hbond_median_ps']
        hb_med_delta = hb_med_d - hb_med_w if pd.notna(hb_med_d) and pd.notna(hb_med_w) else np.nan
        
        # ── DES component coordination ────────────────────────────
        # Collect whatever DES coordination columns exist for this row
        des_coord = {}
        for comp in ['CHO', 'CLA', 'URE', 'GOL']:
            col = f'{comp}_coord_mean'
            if col in des_row.index and pd.notna(des_row[col]):
                des_coord[comp] = des_row[col]
        
        delta_rows.append({
            'peptide': pep,
            'solvent': des,
            'comparison': f'{des} vs water',
            # SASA
            'SASA_water': round(water['SASA_mean_nm2'], 5),
            'SASA_DES': round(des_row['SASA_mean_nm2'], 5),
            'dSASA': round(sasa_delta, 5),
            'dSASA_CI95_lo': round(sasa_delta - sasa_delta_hw, 5),
            'dSASA_CI95_hi': round(sasa_delta + sasa_delta_hw, 5),
            'dSASA_pct': round(sasa_pct, 1),
            'SASA_effect_size': round(sasa_es, 2),
            # Water coordination
            'WaterCoord_water': round(water['water_coord_mean'], 4),
            'WaterCoord_DES': round(des_row['water_coord_mean'], 4),
            'dWaterCoord': round(wc_delta, 4),
            'dWaterCoord_CI95_lo': round(wc_delta - wc_delta_hw, 4),
            'dWaterCoord_CI95_hi': round(wc_delta + wc_delta_hw, 4),
            'dWaterCoord_pct': round(wc_pct, 1),
            'WaterCoord_effect_size': round(wc_es, 2),
            # H-bond events
            'Hbond_events_water': hb_water,
            'Hbond_events_DES': hb_des,
            'dHbond_events': hb_delta,
            'dHbond_events_pct': round(hb_pct, 1) if pd.notna(hb_pct) else None,
            # H-bond lifetime
            'Hbond_median_water_ps': hb_med_w if pd.notna(hb_med_w) else None,
            'Hbond_median_DES_ps': hb_med_d if pd.notna(hb_med_d) else None,
            'dHbond_median_ps': round(hb_med_delta, 1) if pd.notna(hb_med_delta) else None,
            # DES component coordination
            'CHO_coord': des_coord.get('CHO'),
            'CLA_coord': des_coord.get('CLA'),
            'URE_coord': des_coord.get('URE'),
            'GOL_coord': des_coord.get('GOL'),
        })

delta_df = pd.DataFrame(delta_rows)

# Save
delta_csv = os.path.join(ANALYSIS_DIR, 'comparative_deltas.csv')
delta_df.to_csv(delta_csv, index=False)
print(f'Saved: {delta_csv}')
print(f'Rows: {len(delta_df)}')

### 2b. Delta summary table

In [ ]:
print('=' * 90)
print('CROSS-SOLVENT DELTAS: DES vs Water')
print('=' * 90)

print('\n--- SASA (nm\u00b2) ---')
print(delta_df[['peptide', 'solvent', 'SASA_water', 'SASA_DES',
                'dSASA', 'dSASA_CI95_lo', 'dSASA_CI95_hi',
                'dSASA_pct', 'SASA_effect_size']].to_string(index=False))

print('\n--- Water Coordination ---')
print(delta_df[['peptide', 'solvent', 'WaterCoord_water', 'WaterCoord_DES',
                'dWaterCoord', 'dWaterCoord_CI95_lo', 'dWaterCoord_CI95_hi',
                'dWaterCoord_pct', 'WaterCoord_effect_size']].to_string(index=False))

print('\n--- H-bond Events ---')
print(delta_df[['peptide', 'solvent', 'Hbond_events_water', 'Hbond_events_DES',
                'dHbond_events', 'dHbond_events_pct']].to_string(index=False))

print('\n--- H-bond Median Lifetime (ps) ---')
print(delta_df[['peptide', 'solvent', 'Hbond_median_water_ps',
                'Hbond_median_DES_ps', 'dHbond_median_ps']].to_string(index=False))

print('\n--- DES Component Coordination (mean contacts within 0.35 nm) ---')
print(delta_df[['peptide', 'solvent', 'CHO_coord', 'CLA_coord',
                'URE_coord', 'GOL_coord']].to_string(index=False))

---
## 3. Cross-motif comparison

Rank the three motifs by their responsiveness to DES solvents.  
We define responsiveness as the mean fractional SASA change across  
both DES solvents, since SASA is the most directly interpretable  
metric for motif accessibility.

In [ ]:
# ── Aggregate responsiveness per motif ──────────────────────────────
motif_summary = []

for pep in PEPTIDES:
    pep_deltas = delta_df[delta_df['peptide'] == pep]
    
    # Mean SASA change across both DES solvents
    mean_dsasa = pep_deltas['dSASA'].mean()
    mean_dsasa_pct = pep_deltas['dSASA_pct'].mean()
    mean_sasa_es = pep_deltas['SASA_effect_size'].mean()
    
    # Mean water coordination change
    mean_dwc = pep_deltas['dWaterCoord'].mean()
    mean_dwc_pct = pep_deltas['dWaterCoord_pct'].mean()
    mean_wc_es = pep_deltas['WaterCoord_effect_size'].mean()
    
    # Reline vs glyceline differential
    rel_row = pep_deltas[pep_deltas['solvent'] == 'reline'].iloc[0]
    gly_row = pep_deltas[pep_deltas['solvent'] == 'glyceline'].iloc[0]
    reline_glyceline_diff_sasa = gly_row['dSASA'] - rel_row['dSASA']
    
    motif_summary.append({
        'peptide': pep,
        'mean_dSASA_nm2': round(mean_dsasa, 4),
        'mean_dSASA_pct': round(mean_dsasa_pct, 1),
        'mean_SASA_effect_size': round(mean_sasa_es, 2),
        'mean_dWaterCoord': round(mean_dwc, 4),
        'mean_dWaterCoord_pct': round(mean_dwc_pct, 1),
        'mean_WaterCoord_effect_size': round(mean_wc_es, 2),
        'glyceline_minus_reline_dSASA': round(reline_glyceline_diff_sasa, 4),
        'reline_dSASA': round(rel_row['dSASA'], 4),
        'glyceline_dSASA': round(gly_row['dSASA'], 4),
    })

motif_df = pd.DataFrame(motif_summary)
motif_df = motif_df.sort_values('mean_dSASA_pct', ascending=False).reset_index(drop=True)

motif_csv = os.path.join(ANALYSIS_DIR, 'cross_motif_summary.csv')
motif_df.to_csv(motif_csv, index=False)

print(f'Saved: {motif_csv}')
print()
print('=== MOTIF RESPONSIVENESS RANKING (by mean % SASA change in DES) ===')
print(motif_df.to_string(index=False))

print('\n\nInterpretation:')
top = motif_df.iloc[0]
print(f'  Most responsive motif: {top["peptide"]} '
      f'(mean SASA increase: {top["mean_dSASA_pct"]:.1f}%)')
print(f'  glyceline - reline differential SASA:')
for _, row in motif_df.iterrows():
    direction = 'glyceline > reline' if row['glyceline_minus_reline_dSASA'] > 0 else 'reline > glyceline'
    print(f'    {row["peptide"]}: {row["glyceline_minus_reline_dSASA"]:+.4f} nm\u00b2 ({direction})')

---
## 4. Publication figures

### 4a. ΔSASA bar chart — grouped by peptide, coloured by DES solvent

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
bar_width = 0.3
x = np.arange(len(PEPTIDES))

for k, des in enumerate(DES_SOLVENTS):
    subset = delta_df[delta_df['solvent'] == des].sort_values('peptide')
    # Ensure peptide order matches PEPTIDES list
    subset = subset.set_index('peptide').loc[PEPTIDES].reset_index()
    
    vals = subset['dSASA'].values
    lo = subset['dSASA_CI95_lo'].values
    hi = subset['dSASA_CI95_hi'].values
    err_lo = vals - lo
    err_hi = hi - vals
    
    ax.bar(x + k * bar_width, vals, bar_width,
           yerr=[err_lo, err_hi],
           label=des.capitalize(),
           color=SOLVENT_COLORS[des],
           edgecolor='white', linewidth=0.5,
           capsize=3, error_kw={'linewidth': 0.8})

ax.axhline(y=0, color='grey', linestyle='-', linewidth=0.5)
ax.set_xlabel('Peptide motif')
ax.set_ylabel('\u0394SASA (DES \u2212 water) (nm\u00b2)')
ax.set_title('Change in Motif Solvent Accessible Surface Area\nDES vs Water')
ax.set_xticks(x + bar_width / 2)
ax.set_xticklabels(PEPTIDES)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'delta_SASA_bar.png'), dpi=200)
plt.show()
print(f'Saved: {os.path.join(FIGURES_DIR, "delta_SASA_bar.png")}')

### 4b. ΔWater coordination bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
bar_width = 0.3
x = np.arange(len(PEPTIDES))

for k, des in enumerate(DES_SOLVENTS):
    subset = delta_df[delta_df['solvent'] == des].set_index('peptide').loc[PEPTIDES].reset_index()
    
    vals = subset['dWaterCoord'].values
    lo = subset['dWaterCoord_CI95_lo'].values
    hi = subset['dWaterCoord_CI95_hi'].values
    err_lo = vals - lo
    err_hi = hi - vals
    
    ax.bar(x + k * bar_width, vals, bar_width,
           yerr=[err_lo, err_hi],
           label=des.capitalize(),
           color=SOLVENT_COLORS[des],
           edgecolor='white', linewidth=0.5,
           capsize=3, error_kw={'linewidth': 0.8})

ax.axhline(y=0, color='grey', linestyle='-', linewidth=0.5)
ax.set_xlabel('Peptide motif')
ax.set_ylabel('\u0394Water coordination (DES \u2212 water)')
ax.set_title('Change in Water Coordination at Motif Backbone O\nDES vs Water')
ax.set_xticks(x + bar_width / 2)
ax.set_xticklabels(PEPTIDES)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'delta_water_coordination_bar.png'), dpi=200)
plt.show()

### 4c. Summary heatmap — all metrics × all conditions

Rows: peptide × DES solvent (6 comparisons)  
Columns: ΔSASA%, ΔWaterCoord%, ΔH-bond events%, effect sizes  
Colour: magnitude of change (diverging colourmap)

In [ ]:
# Build heatmap matrix
heatmap_cols = ['dSASA_pct', 'dWaterCoord_pct', 'SASA_effect_size', 'WaterCoord_effect_size']
col_labels = ['\u0394SASA (%)', '\u0394Water coord (%)', 'SASA effect size', 'Water coord\neffect size']

row_labels = []
heatmap_data = []

for pep in PEPTIDES:
    for des in DES_SOLVENTS:
        row = delta_df[(delta_df['peptide'] == pep) & (delta_df['solvent'] == des)].iloc[0]
        row_labels.append(f'{pep} / {des}')
        heatmap_data.append([row[c] for c in heatmap_cols])

heatmap_arr = np.array(heatmap_data)

fig, ax = plt.subplots(figsize=(8, 5))

# Use diverging colourmap centred at 0 for % changes, sequential for effect sizes
# Since we're mixing scales, normalise each column to [0, 1] for display
# but show raw values as annotations
im = ax.imshow(heatmap_arr, aspect='auto', cmap='RdYlBu_r')

ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=9)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=9)

# Annotate cells with values
for i in range(heatmap_arr.shape[0]):
    for j in range(heatmap_arr.shape[1]):
        val = heatmap_arr[i, j]
        fmt = f'{val:+.1f}' if j < 2 else f'{val:.1f}'
        text_color = 'white' if abs(val) > np.nanmax(abs(heatmap_arr)) * 0.6 else 'black'
        ax.text(j, i, fmt, ha='center', va='center', fontsize=9, color=text_color)

ax.set_title('Comparative Summary: DES vs Water Effects', fontsize=11)
fig.colorbar(im, ax=ax, shrink=0.8, label='Value')
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'summary_heatmap.png'), dpi=200)
plt.show()

### 4d. DES component coordination breakdown

Stacked bar chart showing the relative contribution of each DES component  
to the total non-water coordination at the motif.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

comp_colors = {
    'CHO': '#ff7f0e',  # choline — orange
    'CLA': '#9467bd',  # chloride — purple
    'URE': '#d62728',  # urea — red
    'GOL': '#2ca02c',  # glycerol — green
}

for ax, des in zip(axes, DES_SOLVENTS):
    subset = delta_df[delta_df['solvent'] == des].set_index('peptide').loc[PEPTIDES]
    
    if des == 'reline':
        components = ['CHO', 'CLA', 'URE']
    else:
        components = ['CHO', 'CLA', 'GOL']
    
    x = np.arange(len(PEPTIDES))
    bottom = np.zeros(len(PEPTIDES))
    
    for comp in components:
        col = f'{comp}_coord'
        vals = subset[col].fillna(0).values.astype(float)
        ax.bar(x, vals, 0.5, bottom=bottom,
               label=comp, color=comp_colors[comp],
               edgecolor='white', linewidth=0.5)
        # Annotate non-trivial values
        for xi, v, b in zip(x, vals, bottom):
            if v > 0.005:
                ax.text(xi, b + v / 2, f'{v:.3f}',
                        ha='center', va='center', fontsize=7)
        bottom += vals
    
    ax.set_title(f'{des.capitalize()}', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(PEPTIDES)
    ax.set_xlabel('Peptide motif')
    ax.legend(fontsize=8, loc='upper right')

axes[0].set_ylabel('Mean DES component contacts\nwithin 0.35 nm of motif backbone O')
fig.suptitle('DES Component Coordination at Motif', fontsize=12, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'des_component_coordination.png'),
            dpi=200, bbox_inches='tight')
plt.show()

### 4e. Per-residue SASA comparison across solvents

For each peptide, show the per-residue SASA in all three solvents  
to identify which specific residues within the motif are most affected.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, pep in zip(axes, PEPTIDES):
    pep_res = df_res[df_res['peptide'] == pep].copy()
    
    # Get residue labels from water system (same for all solvents)
    water_res = pep_res[pep_res['solvent'] == 'water'].sort_values('residue_idx')
    res_labels = water_res['residue'].values
    n_res = len(res_labels)
    x = np.arange(n_res)
    bar_width = 0.25
    
    for k, solv in enumerate(SOLVENTS):
        solv_res = pep_res[pep_res['solvent'] == solv].sort_values('residue_idx')
        vals = solv_res['SASA_mean_nm2'].values
        lo = solv_res['SASA_CI95_lo'].values
        hi = solv_res['SASA_CI95_hi'].values
        err_lo = vals - lo
        err_hi = hi - vals
        
        ax.bar(x + k * bar_width, vals, bar_width,
               yerr=[err_lo, err_hi],
               label=solv.capitalize(),
               color=SOLVENT_COLORS[solv],
               edgecolor='white', linewidth=0.5,
               capsize=2, error_kw={'linewidth': 0.6})
    
    ax.set_title(f'{pep} motif', fontsize=11)
    ax.set_xticks(x + bar_width)
    ax.set_xticklabels(res_labels, rotation=45, ha='right', fontsize=8)
    ax.set_xlabel('Residue')
    if ax == axes[0]:
        ax.set_ylabel('SASA (nm\u00b2)')
    ax.legend(fontsize=7)

fig.suptitle('Per-Residue SASA by Solvent', fontsize=12, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'per_residue_sasa_comparison.png'),
            dpi=200, bbox_inches='tight')
plt.show()

### 4f. Combined overview — SASA and coordination side by side

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
bar_width = 0.25
x = np.arange(len(PEPTIDES))

# Panel A: Absolute SASA by solvent
for k, solv in enumerate(SOLVENTS):
    subset = df[df['solvent'] == solv].set_index('peptide').loc[PEPTIDES]
    vals = subset['SASA_mean_nm2'].values
    lo = subset['SASA_CI95_lo'].values
    hi = subset['SASA_CI95_hi'].values
    
    ax1.bar(x + k * bar_width, vals, bar_width,
            yerr=[vals - lo, hi - vals],
            label=solv.capitalize(),
            color=SOLVENT_COLORS[solv],
            edgecolor='white', linewidth=0.5,
            capsize=3, error_kw={'linewidth': 0.8})

ax1.set_xlabel('Peptide motif')
ax1.set_ylabel('Motif SASA (nm\u00b2)')
ax1.set_title('A. Motif Solvent Accessible Surface Area')
ax1.set_xticks(x + bar_width)
ax1.set_xticklabels(PEPTIDES)
ax1.legend(fontsize=8)

# Panel B: Water coordination by solvent
for k, solv in enumerate(SOLVENTS):
    subset = df[df['solvent'] == solv].set_index('peptide').loc[PEPTIDES]
    vals = subset['water_coord_mean'].values
    lo = subset['water_coord_CI95_lo'].values
    hi = subset['water_coord_CI95_hi'].values
    
    ax2.bar(x + k * bar_width, vals, bar_width,
            yerr=[vals - lo, hi - vals],
            label=solv.capitalize(),
            color=SOLVENT_COLORS[solv],
            edgecolor='white', linewidth=0.5,
            capsize=3, error_kw={'linewidth': 0.8})

ax2.set_xlabel('Peptide motif')
ax2.set_ylabel('Water molecules within 0.35 nm')
ax2.set_title('B. Water Coordination at Motif')
ax2.set_xticks(x + bar_width)
ax2.set_xticklabels(PEPTIDES)
ax2.legend(fontsize=8)

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'combined_sasa_coordination.png'), dpi=200)
plt.show()

---
## 5. Interpretive summary

This section synthesises the quantitative observations from the comparative  
analysis. These are observations supported by the data, not conclusions.

In [ ]:
print('=' * 75)
print('STEP 3.2 — INTERPRETIVE SUMMARY')
print('=' * 75)

print('\n1. MOTIF SASA INCREASE IN DES')
print('-' * 40)
for _, row in delta_df.iterrows():
    sig = '*' if row['SASA_effect_size'] > 2 else ''
    print(f'   {row["peptide"]} in {row["solvent"]:>10}: '
          f'\u0394SASA = {row["dSASA"]:+.4f} nm\u00b2 ({row["dSASA_pct"]:+.1f}%) '
          f'ES={row["SASA_effect_size"]:.1f} {sig}')
print('   (* = effect size > 2, indicating non-overlapping CIs)')

print('\n2. WATER COORDINATION REDUCTION IN DES')
print('-' * 40)
for _, row in delta_df.iterrows():
    print(f'   {row["peptide"]} in {row["solvent"]:>10}: '
          f'\u0394WaterCoord = {row["dWaterCoord"]:+.4f} ({row["dWaterCoord_pct"]:+.1f}%) '
          f'ES={row["WaterCoord_effect_size"]:.1f}')

print('\n3. MOTIF RESPONSIVENESS RANKING')
print('-' * 40)
for i, (_, row) in enumerate(motif_df.iterrows()):
    print(f'   {i+1}. {row["peptide"]}: mean \u0394SASA = {row["mean_dSASA_pct"]:+.1f}%, '
          f'mean ES = {row["mean_SASA_effect_size"]:.1f}')

print('\n4. RELINE vs GLYCELINE DIFFERENTIAL')
print('-' * 40)
for _, row in motif_df.iterrows():
    direction = 'glyceline' if row['glyceline_minus_reline_dSASA'] > 0 else 'reline'
    print(f'   {row["peptide"]}: {direction} produces {abs(row["glyceline_minus_reline_dSASA"]):.4f} nm\u00b2 '
          f'greater \u0394SASA')

print('\n5. DES COMPONENT DIRECT CONTACTS')
print('-' * 40)
print('   Urea is the only DES component making substantial direct contact')
print('   with motif backbone O atoms (0.32\u20130.38 contacts within 0.35 nm).')
print('   Choline, chloride, and glycerol contacts are negligible (<0.09).')
print('   This suggests the DES effect on SASA operates through solvent')
print('   restructuring rather than direct DES\u2013peptide binding.')

print('\n6. H-BOND OBSERVATIONS')
print('-' * 40)
print('   - GGE in water: longest-lived backbone H-bonds (median 3 ps vs 2 ps elsewhere)')
print('   - H-bond event count reduced in DES across GGE and CME')
print('   - YIY: no backbone H-bonds in water or glyceline; 1433 events in reline')
print('     (consistent with urea stabilising transient backbone conformations)')

---
## 6. Output verification and next steps

In [ ]:
print('=' * 65)
print('Step 3.2 — Comparative Analysis Complete')
print('=' * 65)

print(f'\nOutput CSVs:')
for f in ['comparative_deltas.csv', 'cross_motif_summary.csv']:
    path = os.path.join(ANALYSIS_DIR, f)
    exists = os.path.exists(path)
    print(f'  [{"OK" if exists else "MISSING"}] {path}')

print(f'\nNew figures:')
new_figs = [
    'delta_SASA_bar.png',
    'delta_water_coordination_bar.png',
    'summary_heatmap.png',
    'des_component_coordination.png',
    'per_residue_sasa_comparison.png',
    'combined_sasa_coordination.png',
]
for f in new_figs:
    path = os.path.join(FIGURES_DIR, f)
    exists = os.path.exists(path)
    print(f'  [{"OK" if exists else "MISSING"}] {f}')

print(f'\nPhase 3 complete.')
print(f'Next: Phase 4 — Downstream computational analysis')
print(f'  Step 4.1: In silico proteolysis')
print(f'  Step 4.2: Bioactivity prediction')